# Multi-Resolution WDM For Links, XYZ, And AET

This notebook applies Wilson-Daubechies-Meyer time-frequency decompositions at multiple resolutions to:

- the six one-way link channels,
- the Michelson `XYZ` TDI channels,
- the orthogonal `AET` TDI channels.

It runs on both noisy-plus-GW products and GW-only products when available. It also builds a first diagnostic for whitening the noisy `A` and `E` maps using the noisy `T` channel as a noise proxy.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample_poly
from wdm_wavelet.wdm import WDM

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "lisa_link_sim":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

BASE_DIR = PROJECT_ROOT / "lisa_link_sim"
OUTPUT_DIR = BASE_DIR / "outputs"
PLOT_DIR = OUTPUT_DIR / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "links_noisy": OUTPUT_DIR / "six_links_noisy.csv",
    "links_gw_only": OUTPUT_DIR / "six_links_gw_only.csv",
    "tdi_noisy": OUTPUT_DIR / "tdi_michelson_XYZ_AET.csv",
    "tdi_gw_only": OUTPUT_DIR / "tdi_gw_only_XYZ_AET.csv",
}

LINK_LABELS = ["12", "23", "31", "13", "32", "21"]
XYZ_LABELS = ["X", "Y", "Z"]
AET_LABELS = ["A", "E", "T"]
GROUP_CHANNELS = {"links": LINK_LABELS, "XYZ": XYZ_LABELS, "AET": AET_LABELS}

FILES


{'links_noisy': PosixPath('/Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/six_links_noisy.csv'),
 'links_gw_only': PosixPath('/Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/six_links_gw_only.csv'),
 'tdi_noisy': PosixPath('/Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/tdi_michelson_XYZ_AET.csv'),
 'tdi_gw_only': PosixPath('/Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/tdi_gw_only_XYZ_AET.csv')}

## Configuration

The WDM resolutions below trade time and frequency detail. Smaller `M` gives finer time localization and coarser frequency bins; larger `M` gives finer frequency localization and coarser time bins.


In [2]:
WDM_CONFIGS = [
    {"label": "dt1_M64", "target_dt": 1.0, "M": 64},
    {"label": "dt1_M128", "target_dt": 1.0, "M": 128},
    {"label": "dt1_M256", "target_dt": 1.0, "M": 256},
    {"label": "dt1_M512", "target_dt": 1.0, "M": 512},
    {"label": "dt1_M1024", "target_dt": 1.0, "M": 1024},
    {"label": "dt1_M2048", "target_dt": 1.0, "M": 2048},
    {"label": "dt1_M4096", "target_dt": 1.0, "M": 4096},
    {"label": "dt2_M1024", "target_dt": 2.0, "M": 1024},
    {"label": "dt2_M2048", "target_dt": 2.0, "M": 2048},
    {"label": "dt2_M4096", "target_dt": 2.0, "M": 4096},
    {"label": "dt5_M2048", "target_dt": 5.0, "M": 2048},
    {"label": "dt5_M4096", "target_dt": 5.0, "M": 4096},
    {"label": "dt10_M2048", "target_dt": 10.0, "M": 2048},
    {"label": "dt10_M4096", "target_dt": 10.0, "M": 4096},
    {"label": "dt10_M8192", "target_dt": 10.0, "M": 8192},
]

WDM_K = 32
WDM_BETA_ORDER = 6
WDM_PRECISION = 6
WDM_BACKEND = "numba"

LOW_FREQ_CUT = 1e-5
HIGH_FREQ_CUT = 1e-2
SIGNAL_THRESHOLD_FRACTION = 1e-3
OFFSOURCE_GUARD_SECONDS = 500.0
MIN_NOISE_FLOOR = 1e-300

# Common grid for the combined best-pixel plots. Each common TF bin keeps the
# maximum normalized WDM power contributed by any configured resolution.
BEST_PIXEL_TIME_BINS = 240
BEST_PIXEL_FREQ_BINS = 180

SAVE_RAW_POWER = True
SAVE_NORMALIZED_POWER = True
MAKE_GROUP_PLOTS = False
MAKE_COMBINED_BEST_PIXEL_PLOTS = True
MAKE_GRID_PLOTS = True

pd.DataFrame(WDM_CONFIGS)


,label,target_dt,M
0,dt1_M64,1.0,64
1,dt1_M128,1.0,128
2,dt1_M256,1.0,256
3,dt1_M512,1.0,512
4,dt1_M1024,1.0,1024
5,dt1_M2048,1.0,2048
6,dt1_M4096,1.0,4096
7,dt2_M1024,2.0,1024
8,dt2_M2048,2.0,2048
9,dt2_M4096,2.0,4096


## Load And Align Products


In [3]:
def load_products():
    links_noisy_cols = ["sample", "t_waveform", "t_lisa", *[f"noisy_{lab}" for lab in LINK_LABELS]]
    links_gw_cols = ["sample", "t_waveform", "t_lisa", "is_science", *[f"gw_{lab}_hz" for lab in LINK_LABELS]]
    tdi_noisy_cols = ["sample", "t_lisa", "X2", "Y2", "Z2", "A2", "E2", "T2"]
    tdi_gw_cols = ["sample", "t_lisa", "X2_gw_hz", "Y2_gw_hz", "Z2_gw_hz", "A2_gw_hz", "E2_gw_hz", "T2_gw_hz"]

    links_noisy = pd.read_csv(FILES["links_noisy"], usecols=links_noisy_cols)
    links_gw = pd.read_csv(FILES["links_gw_only"], usecols=links_gw_cols)
    links_gw = links_gw[links_gw["is_science"].astype(bool)].reset_index(drop=True)
    tdi_noisy = pd.read_csv(FILES["tdi_noisy"], usecols=tdi_noisy_cols)
    tdi_gw = pd.read_csv(FILES["tdi_gw_only"], usecols=tdi_gw_cols)

    t_lisa = links_noisy["t_lisa"].to_numpy(dtype=float)
    for name, df in [("links_gw", links_gw), ("tdi_noisy", tdi_noisy), ("tdi_gw", tdi_gw)]:
        if len(df) != len(t_lisa):
            raise ValueError(f"{name} length {len(df)} does not match links_noisy length {len(t_lisa)}.")
        if not np.allclose(df["t_lisa"].to_numpy(dtype=float), t_lisa, rtol=0, atol=1e-6):
            raise ValueError(f"{name} is not aligned on the same t_lisa grid.")

    data = {
        "noisy": {
            "links": {lab: links_noisy[f"noisy_{lab}"].to_numpy(dtype=float) for lab in LINK_LABELS},
            "XYZ": {
                "X": tdi_noisy["X2"].to_numpy(dtype=float),
                "Y": tdi_noisy["Y2"].to_numpy(dtype=float),
                "Z": tdi_noisy["Z2"].to_numpy(dtype=float),
            },
            "AET": {ch: tdi_noisy[f"{ch}2"].to_numpy(dtype=float) for ch in AET_LABELS},
        },
        "gw_only": {
            "links": {lab: links_gw[f"gw_{lab}_hz"].to_numpy(dtype=float) for lab in LINK_LABELS},
            "XYZ": {ch: tdi_gw[f"{ch}2_gw_hz"].to_numpy(dtype=float) for ch in XYZ_LABELS},
            "AET": {ch: tdi_gw[f"{ch}2_gw_hz"].to_numpy(dtype=float) for ch in AET_LABELS},
        },
    }
    return t_lisa, data


t_lisa, product_data = load_products()
t_rel = t_lisa - t_lisa[0]
dt = float(np.median(np.diff(t_lisa)))

print(f"samples = {len(t_lisa)}")
print(f"dt      = {dt} s")
print(f"span    = {t_rel[-1] - t_rel[0]:.1f} s")
for source_name in ["noisy", "gw_only"]:
    for group, channels in GROUP_CHANNELS.items():
        values = np.column_stack([product_data[source_name][group][ch] for ch in channels])
        print(f"{source_name:7s} {group:5s}: max abs = {np.nanmax(np.abs(values)):.3e}")


samples = 1000000
dt      = 0.10000000149011612 s
span    = 99999.9 s
noisy   links: max abs = 3.132e+02
noisy   XYZ  : max abs = 2.588e-04
noisy   AET  : max abs = 3.176e-04
gw_only links: max abs = 1.617e-04
gw_only XYZ  : max abs = 2.601e-04
gw_only AET  : max abs = 3.176e-04


## Locate Signal From GW-Only AET


In [4]:
gw_aet_power = sum(product_data["gw_only"]["AET"][ch] ** 2 for ch in AET_LABELS)
signal_peak_idx = int(np.argmax(gw_aet_power))
signal_t_lisa = float(t_lisa[signal_peak_idx])
signal_peak_rel = signal_t_lisa - t_lisa[0]
threshold = SIGNAL_THRESHOLD_FRACTION * float(np.nanmax(gw_aet_power))
signal_indices = np.flatnonzero(gw_aet_power >= threshold)
signal_window_lisa = (float(t_lisa[signal_indices[0]]), float(t_lisa[signal_indices[-1]]))
signal_window_rel = (signal_window_lisa[0] - t_lisa[0], signal_window_lisa[1] - t_lisa[0])

print(f"signal peak rel   = {signal_peak_rel:.1f} s")
print(f"signal window rel = [{signal_window_rel[0]:.1f}, {signal_window_rel[1]:.1f}] s")
print(f"signal duration   = {signal_window_rel[1] - signal_window_rel[0]:.1f} s")


signal peak rel   = 49445.2 s
signal window rel = [45810.2, 51184.0] s
signal duration   = 5373.8 s


## Multi-Resolution WDM Decomposition


In [5]:
def downsample_for_wdm(x, dt, target_dt):
    factor = int(round(target_dt / dt))
    if not np.isclose(factor * dt, target_dt):
        raise ValueError("target_dt must be an integer multiple of the native dt.")
    x = np.asarray(x, dtype=float)
    if factor <= 1:
        return x, dt
    # resample_poly applies an anti-aliasing FIR filter and handles large
    # downsampling factors more robustly than a single decimate call.
    return resample_poly(x, up=1, down=factor), factor * dt


def pad_for_wdm_if_needed(x, wdm):
    """Pad short downsampled series so large-M low-frequency WDM transforms are valid."""
    min_len = max(len(x), 2 * int(wdm.m_H) + 1)
    if len(x) >= min_len:
        return x, 0
    pad_right = min_len - len(x)
    return np.pad(x, (0, pad_right), mode="constant"), pad_right


def wdm_power_map(wdm, x, sample_rate, t0, original_t1):
    x_padded, pad_right = pad_for_wdm_if_needed(x, wdm)
    tf = wdm.t2w(x_padded, sample_rate=sample_rate, t0=t0)
    power = np.abs(tf.data) ** 2
    freqs = np.arange(tf.n_freq, dtype=float) * tf.df
    times_lisa = tf.t0 + (np.arange(tf.n_time, dtype=float) + 0.5) * tf.dt

    # Drop WDM time pixels whose centers fall in the artificial zero padding.
    valid_time = times_lisa <= original_t1
    power = power[:, valid_time]
    times_lisa = times_lisa[valid_time]
    return power, times_lisa, freqs, tf, pad_right


def summarize_map(source_name, group, channel, config, power, normalized_power, times_lisa_wdm, freqs_wdm, tf, pad_right):
    signal_tf_mask = (times_lisa_wdm >= signal_window_lisa[0]) & (times_lisa_wdm <= signal_window_lisa[1])
    offsource_tf_mask = (
        (times_lisa_wdm < signal_window_lisa[0] - OFFSOURCE_GUARD_SECONDS)
        | (times_lisa_wdm > signal_window_lisa[1] + OFFSOURCE_GUARD_SECONDS)
    )
    freq_tf_mask = (freqs_wdm >= LOW_FREQ_CUT) & (freqs_wdm <= HIGH_FREQ_CUT)
    signal_band = normalized_power[np.ix_(freq_tf_mask, signal_tf_mask)]
    offsource_band = normalized_power[np.ix_(freq_tf_mask, offsource_tf_mask)]
    return {
        "source": source_name,
        "group": group,
        "channel": channel,
        "resolution": config["label"],
        "target_dt_s": config["target_dt"],
        "wdm_M": config["M"],
        "wdm_df_hz": float(tf.df),
        "first_positive_freq_hz": float(tf.df),
        "reaches_1e-5_hz": bool(tf.df <= LOW_FREQ_CUT),
        "wdm_dt_tf_s": float(tf.dt),
        "n_time_layers": int(len(times_lisa_wdm)),
        "n_frequency_layers": int(np.sum(freq_tf_mask)),
        "n_signal_time_layers": int(np.sum(signal_tf_mask)),
        "signal_tf_pixels": int(signal_band.size),
        "pad_right_samples": int(pad_right),
        "signal_norm_power_mean": float(np.nanmean(signal_band)) if signal_band.size else np.nan,
        "signal_norm_power_median": float(np.nanmedian(signal_band)) if signal_band.size else np.nan,
        "signal_norm_power_max": float(np.nanmax(signal_band)) if signal_band.size else np.nan,
        "offsource_norm_power_mean": float(np.nanmean(offsource_band)) if offsource_band.size else np.nan,
        "signal_over_offsource_mean": float(np.nanmean(signal_band) / np.nanmean(offsource_band)) if signal_band.size and offsource_band.size else np.nan,
    }


wdm_products = {}
summary_rows = []
npz_payload = {
    "signal_window_lisa": np.asarray(signal_window_lisa, dtype=float),
    "signal_window_rel": np.asarray(signal_window_rel, dtype=float),
    "signal_peak_lisa": np.asarray(signal_t_lisa, dtype=float),
    "signal_peak_rel": np.asarray(signal_peak_rel, dtype=float),
    "resolution_labels": np.asarray([cfg["label"] for cfg in WDM_CONFIGS]),
    "resolution_target_dt_s": np.asarray([cfg["target_dt"] for cfg in WDM_CONFIGS], dtype=float),
    "resolution_M": np.asarray([cfg["M"] for cfg in WDM_CONFIGS], dtype=int),
}

for config in WDM_CONFIGS:
    label = config["label"]
    M = config["M"]
    target_dt = config["target_dt"]
    print(f"Running WDM {label}: target_dt={target_dt} s, M={M}")
    wdm = WDM(M=M, K=WDM_K, beta_order=WDM_BETA_ORDER, precision=WDM_PRECISION, backend=WDM_BACKEND)
    for source_name in ["noisy", "gw_only"]:
        for group, channels in GROUP_CHANNELS.items():
            for ch in channels:
                x_ds, dt_wdm = downsample_for_wdm(product_data[source_name][group][ch], dt, target_dt)
                power, times_lisa_wdm, freqs_wdm, tf, pad_right = wdm_power_map(
                    wdm, x_ds, sample_rate=1.0 / dt_wdm, t0=t_lisa[0], original_t1=t_lisa[-1]
                )
                offsource_tf_mask = (
                    (times_lisa_wdm < signal_window_lisa[0] - OFFSOURCE_GUARD_SECONDS)
                    | (times_lisa_wdm > signal_window_lisa[1] + OFFSOURCE_GUARD_SECONDS)
                )
                noise_floor = np.nanmedian(power[:, offsource_tf_mask], axis=1)
                noise_floor = np.where(np.isfinite(noise_floor) & (noise_floor > 0), noise_floor, MIN_NOISE_FLOOR)
                normalized_power = power / noise_floor[:, None]

                key = (source_name, group, ch, label)
                wdm_products[key] = {
                    "power": power,
                    "normalized_power": normalized_power,
                    "noise_floor": noise_floor,
                    "times_lisa": times_lisa_wdm,
                    "times_rel": times_lisa_wdm - t_lisa[0],
                    "freqs_hz": freqs_wdm,
                    "tf": tf,
                    "config": config,
                    "pad_right": pad_right,
                }
                summary_rows.append(
                    summarize_map(source_name, group, ch, config, power, normalized_power, times_lisa_wdm, freqs_wdm, tf, pad_right)
                )

                prefix = f"{source_name}_{group}_{ch}_{label}"
                if SAVE_RAW_POWER:
                    npz_payload[f"{prefix}_power"] = power.astype(np.float32)
                if SAVE_NORMALIZED_POWER:
                    npz_payload[f"{prefix}_normalized_power"] = normalized_power.astype(np.float32)
                npz_payload[f"{prefix}_noise_floor"] = noise_floor.astype(np.float32)
                npz_payload[f"{prefix}_times_rel"] = (times_lisa_wdm - t_lisa[0]).astype(np.float64)
                npz_payload[f"{prefix}_freqs_hz"] = freqs_wdm.astype(np.float64)

wdm_summary_df = pd.DataFrame(summary_rows)
MULTIRES_WDM_STATS_CSV = OUTPUT_DIR / "multires_wdm_links_xyz_aet_stats.csv"
MULTIRES_WDM_NPZ = OUTPUT_DIR / "multires_wdm_links_xyz_aet_maps.npz"
wdm_summary_df.to_csv(MULTIRES_WDM_STATS_CSV, index=False)
np.savez_compressed(MULTIRES_WDM_NPZ, **npz_payload)

print(f"saved {MULTIRES_WDM_STATS_CSV}")
print(f"saved {MULTIRES_WDM_NPZ}")
wdm_summary_df.head()


Running WDM dt1_M64: target_dt=1.0 s, M=64


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_23148/3708857539.py:122: RuntimeWarning: overflow encountered in cast
  npz_payload[f"{prefix}_normalized_power"] = normalized_power.astype(np.float32)


Running WDM dt1_M128: target_dt=1.0 s, M=128


Running WDM dt1_M256: target_dt=1.0 s, M=256


Running WDM dt1_M512: target_dt=1.0 s, M=512


...

Running WDM dt10_M2048: target_dt=10.0 s, M=2048


Running WDM dt10_M4096: target_dt=10.0 s, M=4096


Running WDM dt10_M8192: target_dt=10.0 s, M=8192


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/multires_wdm_links_xyz_aet_stats.csv
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/multires_wdm_links_xyz_aet_maps.npz


,source,group,channel,resolution,target_dt_s,wdm_M,wdm_df_hz,first_positive_freq_hz,reaches_1e-5_hz,wdm_dt_tf_s,n_time_layers,n_frequency_layers,n_signal_time_layers,signal_tf_pixels,pad_right_samples,signal_norm_power_mean,signal_norm_power_median,signal_norm_power_max,offsource_norm_power_mean,signal_over_offsource_mean
0,noisy,links,12,dt1_M64,1.0,64,0.007812,0.007812,False,63.979528,1563,1,84,84,0,1.610905,1.126807,8.827690,1.431185,1.125574
1,noisy,links,23,dt1_M64,1.0,64,0.007812,0.007812,False,63.979528,1563,1,84,84,0,1.600430,1.113677,9.417622,1.417880,1.128749
2,noisy,links,31,dt1_M64,1.0,64,0.007812,0.007812,False,63.979528,1563,1,84,84,0,11.534216,6.760794,59.348721,3.057790,3.772076
3,noisy,links,13,dt1_M64,1.0,64,0.007812,0.007812,False,63.979528,1563,1,84,84,0,1.612724,1.127033,8.844636,1.432756,1.125610
4,noisy,links,32,dt1_M64,1.0,64,0.007812,0.007812,False,63.979528,1563,1,84,84,0,1.600359,1.114613,9.407791,1.417796,1.128765


## T-Channel Whitening For Noisy A And E

This diagnostic uses the noisy `T` channel as a noise proxy for the noisy `A` and `E` WDM maps. For each WDM resolution, we divide `A` and `E` WDM power by the off-source median `T` WDM power in the same frequency layer.


In [6]:
t_whiten_rows = []
t_whiten_payload = {}

for config in WDM_CONFIGS:
    label = config["label"]
    t_prod = wdm_products[("noisy", "AET", "T", label)]
    t_noise_floor = t_prod["noise_floor"]
    for ch in ["A", "E"]:
        prod = wdm_products[("noisy", "AET", ch, label)]
        t_whitened_power = prod["power"] / np.maximum(t_noise_floor[:, None], MIN_NOISE_FLOOR)
        times_lisa_wdm = prod["times_lisa"]
        freqs_wdm = prod["freqs_hz"]
        signal_tf_mask = (times_lisa_wdm >= signal_window_lisa[0]) & (times_lisa_wdm <= signal_window_lisa[1])
        offsource_tf_mask = (
            (times_lisa_wdm < signal_window_lisa[0] - OFFSOURCE_GUARD_SECONDS)
            | (times_lisa_wdm > signal_window_lisa[1] + OFFSOURCE_GUARD_SECONDS)
        )
        freq_tf_mask = (freqs_wdm >= LOW_FREQ_CUT) & (freqs_wdm <= HIGH_FREQ_CUT)
        signal_band = t_whitened_power[np.ix_(freq_tf_mask, signal_tf_mask)]
        offsource_band = t_whitened_power[np.ix_(freq_tf_mask, offsource_tf_mask)]
        t_whiten_rows.append({
            "source": "noisy",
            "group": "AET",
            "channel": ch,
            "whitening_proxy": "T",
            "resolution": label,
            "target_dt_s": config["target_dt"],
            "wdm_M": config["M"],
            "wdm_df_hz": float(prod["tf"].df),
            "first_positive_freq_hz": float(prod["tf"].df),
            "reaches_1e-5_hz": bool(prod["tf"].df <= LOW_FREQ_CUT),
            "wdm_dt_tf_s": float(prod["tf"].dt),
            "n_frequency_layers": int(np.sum(freq_tf_mask)),
            "n_signal_time_layers": int(np.sum(signal_tf_mask)),
            "signal_tf_pixels": int(signal_band.size),
            "t_whiten_signal_mean": float(np.nanmean(signal_band)) if signal_band.size else np.nan,
            "t_whiten_signal_median": float(np.nanmedian(signal_band)) if signal_band.size else np.nan,
            "t_whiten_signal_max": float(np.nanmax(signal_band)) if signal_band.size else np.nan,
            "t_whiten_offsource_mean": float(np.nanmean(offsource_band)) if offsource_band.size else np.nan,
            "t_whiten_signal_over_offsource_mean": float(np.nanmean(signal_band) / np.nanmean(offsource_band)) if signal_band.size and offsource_band.size else np.nan,
        })
        key = f"noisy_AET_{ch}_Twhiten_{label}"
        t_whiten_payload[f"{key}_power"] = t_whitened_power.astype(np.float32)
        t_whiten_payload[f"{key}_times_rel"] = prod["times_rel"].astype(np.float64)
        t_whiten_payload[f"{key}_freqs_hz"] = freqs_wdm.astype(np.float64)

t_whiten_df = pd.DataFrame(t_whiten_rows)
T_WHITEN_STATS_CSV = OUTPUT_DIR / "multires_wdm_AE_Twhiten_stats.csv"
T_WHITEN_NPZ = OUTPUT_DIR / "multires_wdm_AE_Twhiten_maps.npz"
t_whiten_df.to_csv(T_WHITEN_STATS_CSV, index=False)
np.savez_compressed(T_WHITEN_NPZ, **t_whiten_payload)

print(f"saved {T_WHITEN_STATS_CSV}")
print(f"saved {T_WHITEN_NPZ}")
t_whiten_df


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/multires_wdm_AE_Twhiten_stats.csv
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/multires_wdm_AE_Twhiten_maps.npz


,source,group,channel,whitening_proxy,resolution,target_dt_s,wdm_M,wdm_df_hz,first_positive_freq_hz,reaches_1e-5_hz,wdm_dt_tf_s,n_frequency_layers,n_signal_time_layers,signal_tf_pixels,t_whiten_signal_mean,t_whiten_signal_median,t_whiten_signal_max,t_whiten_offsource_mean,t_whiten_signal_over_offsource_mean
0,noisy,AET,A,T,dt1_M64,1.0,64,0.007812,0.007812,False,63.979528,1,84,84,20728.006558,642.511164,441145.374510,34.971777,592.706693
1,noisy,AET,E,T,dt1_M64,1.0,64,0.007812,0.007812,False,63.979528,1,84,84,19155.653029,630.705255,405321.621172,56.257312,340.500679
2,noisy,AET,A,T,dt1_M128,1.0,128,0.003906,0.003906,False,127.877240,2,42,84,27479.802152,652.142280,632879.553678,46.430268,591.851039
3,noisy,AET,E,T,dt1_M128,1.0,128,0.003906,0.003906,False,127.877240,2,42,84,26164.908300,613.260436,607620.205224,84.207639,310.718939
4,noisy,AET,A,T,dt1_M256,1.0,256,0.001953,0.001953,False,255.754480,5,21,105,30329.806077,795.067009,616957.004055,51.559414,588.249631
5,noisy,AET,E,T,dt1_M256,1.0,256,0.001953,0.001953,False,255.754480,5,21,105,28912.828775,780.875533,586066.316211,95.331213,303.288167
6,noisy,AET,A,T,dt1_M512,1.0,512,0.000977,0.000977,False,510.204089,10,10,100,18821.352419,1013.998136,272238.347948,57.461250,327.548608
7,noisy,AET,E,T,dt1_M512,1.0,512,0.000977,0.000977,False,510.204089,10,10,100,17279.459560,890.177916,272629.154500,109.940510,157.170996
8,noisy,AET,A,T,dt1_M1024,1.0,1024,0.000488,0.000488,False,1020.408178,20,5,100,24988.271183,1259.140974,301384.659997,59.418015,420.550418
9,noisy,AET,E,T,dt1_M1024,1.0,1024,0.000488,0.000488,False,1020.408178,20,5,100,23775.705853,1630.554638,301528.865632,117.425698,202.474469


## Multi-Resolution Plots


In [7]:
def _centers_to_edges(values):
    values = np.asarray(values, dtype=float)
    if values.size == 1:
        width = 1.0
        return np.array([values[0] - 0.5 * width, values[0] + 0.5 * width])
    edges = np.empty(values.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (values[:-1] + values[1:])
    edges[0] = values[0] - 0.5 * (values[1] - values[0])
    edges[-1] = values[-1] + 0.5 * (values[-1] - values[-2])
    return edges


def _tf_plot_edges(times_rel_wdm, freqs, tf_dt):
    time_edges = np.concatenate([
        [times_rel_wdm[0] - 0.5 * tf_dt],
        0.5 * (times_rel_wdm[:-1] + times_rel_wdm[1:]),
        [times_rel_wdm[-1] + 0.5 * tf_dt],
    ])
    freq_edges = _centers_to_edges(freqs)
    freq_edges = np.maximum(freq_edges, np.nextafter(0.0, 1.0))
    return time_edges, freq_edges


def _robust_limits(arrays, low=2.0, high=98.0):
    finite = []
    for arr in arrays:
        vals = np.asarray(arr)[np.isfinite(arr)]
        if vals.size:
            finite.append(vals)
    if not finite:
        return -1.0, 1.0
    vals = np.concatenate(finite)
    vmin, vmax = np.nanpercentile(vals, [low, high])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin = float(np.nanmin(vals))
        vmax = float(np.nanmax(vals))
    if vmin == vmax:
        vmin -= 1.0
        vmax += 1.0
    return float(vmin), float(vmax)


def _log_power(power):
    return np.log10(np.asarray(power, dtype=float) + 1e-30)


def _resolution_title(config):
    return f"{config['label']}\ndt={config['target_dt']:g}s, M={config['M']}"


def plot_multires_grid(source_name, group):
    channels = GROUP_CHANNELS[group]
    nrows = len(WDM_CONFIGS)
    ncols = len(channels)
    prepared = {}
    images_for_limits = []

    for r, config in enumerate(WDM_CONFIGS):
        label = config["label"]
        for c, ch in enumerate(channels):
            prod = wdm_products[(source_name, group, ch, label)]
            freqs = prod["freqs_hz"]
            fmask = (freqs > 0.0) & (freqs <= HIGH_FREQ_CUT)
            image = _log_power(prod["normalized_power"][fmask])
            time_edges, freq_edges = _tf_plot_edges(prod["times_rel"], freqs[fmask], prod["tf"].dt)
            prepared[(r, c)] = (time_edges, freq_edges, image)
            images_for_limits.append(image)

    vmin, vmax = _robust_limits(images_for_limits)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(max(3.2 * ncols, 9), max(1.35 * nrows, 12)),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    axes = np.asarray(axes).reshape(nrows, ncols)
    last_im = None

    for r, config in enumerate(WDM_CONFIGS):
        for c, ch in enumerate(channels):
            ax = axes[r, c]
            time_edges, freq_edges, image = prepared[(r, c)]
            last_im = ax.pcolormesh(
                time_edges,
                freq_edges,
                image,
                shading="auto",
                cmap="magma",
                vmin=vmin,
                vmax=vmax,
            )
            ax.set_yscale("log")
            ax.set_ylim(LOW_FREQ_CUT, HIGH_FREQ_CUT)
            if r == 0:
                ax.set_title(ch, fontsize=10)
            if c == 0:
                ax.set_ylabel(_resolution_title(config), fontsize=8)
            else:
                ax.tick_params(labelleft=False)
            if r < nrows - 1:
                ax.tick_params(labelbottom=False)
            else:
                ax.set_xlabel("time [s]")

    cbar = fig.colorbar(last_im, ax=axes, shrink=0.85, extend="both")
    cbar.set_label(r"$\log_{10}$ off-source-normalized WDM power")
    fig.suptitle(
        f"Multi-resolution WDM grid: {source_name} {group}\n"
        f"frequency range {LOW_FREQ_CUT:.0e}-{HIGH_FREQ_CUT:.0e} Hz; robust color range [{vmin:.2f}, {vmax:.2f}]",
        fontsize=12,
    )
    out = PLOT_DIR / f"15_multires_grid_{source_name}_{group}.png"
    fig.savefig(out, dpi=160)
    plt.show()
    print(f"saved {out}")


if MAKE_GRID_PLOTS:
    for source_name in ["noisy", "gw_only"]:
        for group in ["links", "XYZ", "AET"]:
            plot_multires_grid(source_name, group)


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_23148/3648444974.py:116: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/15_multires_grid_noisy_links.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/15_multires_grid_noisy_XYZ.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/15_multires_grid_noisy_AET.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/15_multires_grid_gw_only_links.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/15_multires_grid_gw_only_XYZ.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/15_multires_grid_gw_only_AET.png


## T-Whitened A/E Plots


In [8]:
def plot_t_whitened_grid():
    channels = ["A", "E"]
    nrows = len(WDM_CONFIGS)
    ncols = len(channels)
    prepared = {}
    images_for_limits = []

    for r, config in enumerate(WDM_CONFIGS):
        label = config["label"]
        for c, ch in enumerate(channels):
            prod = wdm_products[("noisy", "AET", ch, label)]
            key = f"noisy_AET_{ch}_Twhiten_{label}_power"
            freqs = prod["freqs_hz"]
            fmask = (freqs > 0.0) & (freqs <= HIGH_FREQ_CUT)
            image = _log_power(t_whiten_payload[key][fmask])
            time_edges, freq_edges = _tf_plot_edges(prod["times_rel"], freqs[fmask], prod["tf"].dt)
            prepared[(r, c)] = (time_edges, freq_edges, image)
            images_for_limits.append(image)

    vmin, vmax = _robust_limits(images_for_limits)
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(8, max(1.35 * nrows, 12)),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )
    axes = np.asarray(axes).reshape(nrows, ncols)
    last_im = None

    for r, config in enumerate(WDM_CONFIGS):
        for c, ch in enumerate(channels):
            ax = axes[r, c]
            time_edges, freq_edges, image = prepared[(r, c)]
            last_im = ax.pcolormesh(
                time_edges,
                freq_edges,
                image,
                shading="auto",
                cmap="viridis",
                vmin=vmin,
                vmax=vmax,
            )
            ax.set_yscale("log")
            ax.set_ylim(LOW_FREQ_CUT, HIGH_FREQ_CUT)
            if r == 0:
                ax.set_title(f"{ch} / T-noise proxy", fontsize=10)
            if c == 0:
                ax.set_ylabel(_resolution_title(config), fontsize=8)
            else:
                ax.tick_params(labelleft=False)
            if r < nrows - 1:
                ax.tick_params(labelbottom=False)
            else:
                ax.set_xlabel("time [s]")

    cbar = fig.colorbar(last_im, ax=axes, shrink=0.85, extend="both")
    cbar.set_label(r"$\log_{10}$ T-whitened WDM power")
    fig.suptitle(
        f"Multi-resolution WDM grid: noisy A/E whitened by T\n"
        f"frequency range {LOW_FREQ_CUT:.0e}-{HIGH_FREQ_CUT:.0e} Hz; robust color range [{vmin:.2f}, {vmax:.2f}]",
        fontsize=12,
    )
    out = PLOT_DIR / "15_multires_grid_noisy_AE_Twhiten.png"
    fig.savefig(out, dpi=160)
    plt.show()
    print(f"saved {out}")


if MAKE_GRID_PLOTS:
    plot_t_whitened_grid()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/15_multires_grid_noisy_AE_Twhiten.png


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_23148/3739068912.py:67: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Combined Best-Pixel Multi-Resolution Maps

Each common time-frequency bin keeps the highest normalized WDM power from any configured resolution. This gives a compact multi-resolution view: high-frequency/time-local structure can come from small `M`, while the lowest-frequency bins can come from low-rate/high-`M` configurations that actually reach `1e-5 Hz`.


In [9]:
def combined_best_pixel_map(source_name, group, channel, *, whitened_by_T=False):
    time_edges = np.linspace(t_rel[0], t_rel[-1], BEST_PIXEL_TIME_BINS + 1)
    freq_edges = np.geomspace(LOW_FREQ_CUT, HIGH_FREQ_CUT, BEST_PIXEL_FREQ_BINS + 1)
    best = np.full((BEST_PIXEL_FREQ_BINS, BEST_PIXEL_TIME_BINS), np.nan, dtype=float)
    best_resolution = np.full((BEST_PIXEL_FREQ_BINS, BEST_PIXEL_TIME_BINS), "", dtype=object)

    for config in WDM_CONFIGS:
        label = config["label"]
        if whitened_by_T:
            prod = wdm_products[("noisy", "AET", channel, label)]
            power = t_whiten_payload[f"noisy_AET_{channel}_Twhiten_{label}_power"]
        else:
            prod = wdm_products[(source_name, group, channel, label)]
            power = prod["normalized_power"]
        freqs = prod["freqs_hz"]
        times = prod["times_rel"]
        fmask = (freqs >= LOW_FREQ_CUT) & (freqs <= HIGH_FREQ_CUT)
        if not np.any(fmask) or len(times) == 0:
            continue
        values = power[fmask]
        freq_centers = freqs[fmask]

        tt = np.tile(times, len(freq_centers))
        ff = np.repeat(freq_centers, len(times))
        vv = values.reshape(-1)
        ti = np.digitize(tt, time_edges) - 1
        fi = np.digitize(ff, freq_edges) - 1
        valid = np.isfinite(vv) & (ti >= 0) & (ti < BEST_PIXEL_TIME_BINS) & (fi >= 0) & (fi < BEST_PIXEL_FREQ_BINS)
        for f_idx, t_idx, val in zip(fi[valid], ti[valid], vv[valid]):
            if np.isnan(best[f_idx, t_idx]) or val > best[f_idx, t_idx]:
                best[f_idx, t_idx] = val
                best_resolution[f_idx, t_idx] = label

    return time_edges, freq_edges, best, best_resolution


def summarize_combined_best(source_name, group, channel, best, best_resolution):
    if not np.any(np.isfinite(best)):
        return {
            "source": source_name, "group": group, "channel": channel,
            "best_pixel_power": np.nan, "best_pixel_resolution": "", "finite_bins": 0,
        }
    idx = np.nanargmax(best)
    f_idx, t_idx = np.unravel_index(idx, best.shape)
    return {
        "source": source_name,
        "group": group,
        "channel": channel,
        "best_pixel_power": float(best[f_idx, t_idx]),
        "best_pixel_log10_power": float(np.log10(best[f_idx, t_idx] + 1e-30)),
        "best_pixel_resolution": best_resolution[f_idx, t_idx],
        "finite_bins": int(np.sum(np.isfinite(best))),
    }


combined_best_rows = []
combined_payload = {
    "time_edges_rel": np.linspace(t_rel[0], t_rel[-1], BEST_PIXEL_TIME_BINS + 1),
    "freq_edges_hz": np.geomspace(LOW_FREQ_CUT, HIGH_FREQ_CUT, BEST_PIXEL_FREQ_BINS + 1),
}


def plot_combined_group(source_name, group):
    channels = GROUP_CHANNELS[group]
    fig, axes = plt.subplots(1, len(channels), figsize=(4.2 * len(channels), 3.8), sharey=True)
    if len(channels) == 1:
        axes = [axes]
    maps = {}
    images_for_limits = []
    for ch in channels:
        time_edges, freq_edges, best, best_resolution = combined_best_pixel_map(source_name, group, ch)
        maps[ch] = (time_edges, freq_edges, best, best_resolution)
        images_for_limits.append(_log_power(best))
    vmin, vmax = _robust_limits(images_for_limits)
    last_im = None
    for ax, ch in zip(axes, channels):
        time_edges, freq_edges, best, best_resolution = maps[ch]
        combined_best_rows.append(summarize_combined_best(source_name, group, ch, best, best_resolution))
        prefix = f"{source_name}_{group}_{ch}_bestpixel"
        combined_payload[f"{prefix}_power"] = best.astype(np.float32)
        combined_payload[f"{prefix}_resolution"] = best_resolution.astype(str)
        image = np.log10(best + 1e-30)
        last_im = ax.pcolormesh(time_edges, freq_edges, image, shading="auto", cmap="magma", vmin=vmin, vmax=vmax)
        ax.set_yscale("log")
        ax.set_ylim(LOW_FREQ_CUT, HIGH_FREQ_CUT)
        ax.set_title(ch)
        ax.set_xlabel("time [s]")
    axes[0].set_ylabel("frequency [Hz]")
    cbar = fig.colorbar(last_im, ax=axes, label=r"$\log_{10}$ best normalized WDM power", extend="both")
    fig.suptitle(f"Combined best-pixel WDM: {source_name} {group}")
    fig.tight_layout()
    out = PLOT_DIR / f"14_bestpixel_wdm_{source_name}_{group}.png"
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"saved {out}")


if MAKE_COMBINED_BEST_PIXEL_PLOTS:
    for source_name in ["noisy", "gw_only"]:
        for group in ["links", "XYZ", "AET"]:
            plot_combined_group(source_name, group)

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)
    maps = {}
    images_for_limits = []
    for ch in ["A", "E"]:
        time_edges, freq_edges, best, best_resolution = combined_best_pixel_map("noisy", "AET", ch, whitened_by_T=True)
        maps[ch] = (time_edges, freq_edges, best, best_resolution)
        images_for_limits.append(_log_power(best))
    vmin, vmax = _robust_limits(images_for_limits)
    last_im = None
    for ax, ch in zip(axes, ["A", "E"]):
        time_edges, freq_edges, best, best_resolution = maps[ch]
        combined_best_rows.append(summarize_combined_best("noisy_Twhiten", "AET", ch, best, best_resolution))
        prefix = f"noisy_AET_{ch}_Twhiten_bestpixel"
        combined_payload[f"{prefix}_power"] = best.astype(np.float32)
        combined_payload[f"{prefix}_resolution"] = best_resolution.astype(str)
        image = np.log10(best + 1e-30)
        last_im = ax.pcolormesh(time_edges, freq_edges, image, shading="auto", cmap="viridis", vmin=vmin, vmax=vmax)
        ax.set_yscale("log")
        ax.set_ylim(LOW_FREQ_CUT, HIGH_FREQ_CUT)
        ax.set_title(f"{ch} whitened by T")
        ax.set_xlabel("time [s]")
    axes[0].set_ylabel("frequency [Hz]")
    fig.colorbar(last_im, ax=axes, label=r"$\log_{10}$ best T-whitened WDM power", extend="both")
    fig.suptitle("Combined best-pixel WDM: noisy A/E whitened by T")
    fig.tight_layout()
    out = PLOT_DIR / "14_bestpixel_wdm_noisy_AE_Twhiten.png"
    fig.savefig(out, dpi=150)
    plt.show()
    print(f"saved {out}")

combined_best_df = pd.DataFrame(combined_best_rows)
COMBINED_BEST_STATS_CSV = OUTPUT_DIR / "multires_wdm_bestpixel_stats.csv"
COMBINED_BEST_NPZ = OUTPUT_DIR / "multires_wdm_bestpixel_maps.npz"
combined_best_df.to_csv(COMBINED_BEST_STATS_CSV, index=False)
np.savez_compressed(COMBINED_BEST_NPZ, **combined_payload)
print(f"saved {COMBINED_BEST_STATS_CSV}")
print(f"saved {COMBINED_BEST_NPZ}")
combined_best_df


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_23148/3215342478.py:91: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_23148/3215342478.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/14_bestpixel_wdm_noisy_links.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/14_bestpixel_wdm_noisy_XYZ.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/14_bestpixel_wdm_noisy_AET.png


...

saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/14_bestpixel_wdm_gw_only_XYZ.png


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/14_bestpixel_wdm_gw_only_AET.png


/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_23148/3215342478.py:127: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()
/var/folders/p_/x076xwcj62g0046rqqx8v56h0000gn/T/ipykernel_23148/3215342478.py:130: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/plots/14_bestpixel_wdm_noisy_AE_Twhiten.png
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/multires_wdm_bestpixel_stats.csv
saved /Users/shubhanshutiwari/IdeaProjects/LISA-TF/lisa_link_sim/outputs/multires_wdm_bestpixel_maps.npz


,source,group,channel,best_pixel_power,best_pixel_log10_power,best_pixel_resolution,finite_bins
0,noisy,links,12,4.294064e+01,1.632869,dt5_M2048,7594
1,noisy,links,23,2.016012e+02,2.304493,dt10_M4096,7594
2,noisy,links,31,4.744948e+02,2.676231,dt5_M2048,7594
3,noisy,links,13,4.296786e+01,1.633144,dt5_M2048,7594
4,noisy,links,32,1.977827e+02,2.296188,dt10_M4096,7594
5,noisy,links,21,1.834218e+02,2.263451,dt5_M2048,7594
6,noisy,XYZ,X,5.314315e+04,4.725447,dt1_M256,7594
7,noisy,XYZ,Y,1.137890e+04,4.056100,dt1_M256,7594
8,noisy,XYZ,Z,1.536905e+04,4.186647,dt5_M4096,7594
9,noisy,AET,A,2.147351e+04,4.331903,dt1_M256,7594


## Summary Tables


In [10]:
summary_cols = [
    "source", "group", "channel", "resolution", "target_dt_s", "wdm_M", "wdm_df_hz",
    "reaches_1e-5_hz", "wdm_dt_tf_s", "n_frequency_layers", "n_signal_time_layers",
    "signal_over_offsource_mean", "signal_norm_power_mean", "signal_norm_power_max",
]
for group in ["links", "XYZ", "AET"]:
    print(f"\nNoisy {group}: best channels/resolutions")
    display(
        wdm_summary_df[(wdm_summary_df["source"] == "noisy") & (wdm_summary_df["group"] == group)][summary_cols]
        .sort_values("signal_over_offsource_mean", ascending=False)
        .head(15)
        .reset_index(drop=True)
    )

print("\nResolutions that reach 1e-5 Hz")
display(
    wdm_summary_df[["resolution", "target_dt_s", "wdm_M", "wdm_df_hz", "reaches_1e-5_hz", "wdm_dt_tf_s"]]
    .drop_duplicates()
    .sort_values(["reaches_1e-5_hz", "wdm_df_hz"], ascending=[False, True])
    .reset_index(drop=True)
)

print("\nNoisy A/E whitened by T")
display(
    t_whiten_df.sort_values("t_whiten_signal_over_offsource_mean", ascending=False)
    .reset_index(drop=True)
)

print("\nCombined best-pixel summary")
display(
    combined_best_df.sort_values("best_pixel_power", ascending=False).reset_index(drop=True)
)



Noisy links: best channels/resolutions


,source,group,channel,resolution,target_dt_s,wdm_M,wdm_df_hz,reaches_1e-5_hz,wdm_dt_tf_s,n_frequency_layers,n_signal_time_layers,signal_over_offsource_mean,signal_norm_power_mean,signal_norm_power_max
0,noisy,links,31,dt2_M4096,2.0,4096,0.000061,False,7692.307807,163,1,5.752106,18.919318,107.712289
1,noisy,links,21,dt2_M4096,2.0,4096,0.000061,False,7692.307807,163,1,4.184656,11.100508,25.336662
2,noisy,links,31,dt1_M64,1.0,64,0.007812,False,63.979528,1,84,3.772076,11.534216,59.348721
3,noisy,links,31,dt1_M128,1.0,128,0.003906,False,127.877240,2,42,3.586638,11.321957,56.515493
4,noisy,links,21,dt1_M1024,1.0,1024,0.000488,False,1020.408178,20,5,3.569650,11.472235,42.594655
5,noisy,links,31,dt1_M1024,1.0,1024,0.000488,False,1020.408178,20,5,3.356097,11.144957,94.751283
6,noisy,links,31,dt2_M2048,2.0,2048,0.000122,False,4000.000060,81,2,3.208705,10.285000,71.431594
7,noisy,links,21,dt1_M256,1.0,256,0.001953,False,255.754480,5,21,3.193785,10.821841,137.932008
8,noisy,links,31,dt1_M256,1.0,256,0.001953,False,255.754480,5,21,2.889299,9.387701,73.939648
9,noisy,links,21,dt1_M128,1.0,128,0.003906,False,127.877240,2,42,2.877568,10.198593,115.710216



Noisy XYZ: best channels/resolutions


,source,group,channel,resolution,target_dt_s,wdm_M,wdm_df_hz,reaches_1e-5_hz,wdm_dt_tf_s,n_frequency_layers,n_signal_time_layers,signal_over_offsource_mean,signal_norm_power_mean,signal_norm_power_max
0,noisy,XYZ,X,dt2_M4096,2.0,4096,0.000061,False,7692.307807,163,1,1534.391923,6042.868639,20438.598345
1,noisy,XYZ,X,dt1_M256,1.0,256,0.001953,False,255.754480,5,21,1282.058919,2089.986389,53143.151509
2,noisy,XYZ,X,dt1_M128,1.0,128,0.003906,False,127.877240,2,42,1224.593421,1906.121418,37454.394912
3,noisy,XYZ,X,dt1_M4096,1.0,4096,0.000122,False,4000.000060,81,2,1050.402604,2751.382690,17358.029232
4,noisy,XYZ,X,dt1_M64,1.0,64,0.007812,False,63.979528,1,84,1000.942719,1693.188330,35820.455887
5,noisy,XYZ,X,dt2_M2048,2.0,2048,0.000122,False,4000.000060,81,2,928.714316,3373.590340,21923.410972
6,noisy,XYZ,X,dt1_M2048,1.0,2048,0.000244,False,2040.816357,40,3,713.850594,1606.688347,10881.602616
7,noisy,XYZ,X,dt2_M1024,2.0,1024,0.000244,False,2040.816357,40,3,698.272377,1830.388949,17613.249865
8,noisy,XYZ,X,dt1_M1024,1.0,1024,0.000488,False,1020.408178,20,5,665.609542,1307.348046,12838.422685
9,noisy,XYZ,X,dt1_M512,1.0,512,0.000977,False,510.204089,10,10,611.562500,1033.619464,11518.072304



Noisy AET: best channels/resolutions


...

,resolution,target_dt_s,wdm_M,wdm_df_hz,reaches_1e-5_hz,wdm_dt_tf_s
0,dt10_M8192,10.0,8192,0.000006,True,75620.770358
1,dt10_M4096,10.0,4096,0.000012,False,37811.539025
2,dt5_M4096,5.0,4096,0.000024,False,18905.769512
3,dt10_M2048,10.0,2048,0.000024,False,18906.923359
4,dt5_M2048,5.0,2048,0.000049,False,9453.461679
5,dt2_M4096,2.0,4096,0.000061,False,7692.307807
6,dt1_M4096,1.0,4096,0.000122,False,4000.000060
7,dt2_M2048,2.0,2048,0.000122,False,4000.000060
8,dt1_M2048,1.0,2048,0.000244,False,2040.816357
9,dt2_M1024,2.0,1024,0.000244,False,2040.816357



Noisy A/E whitened by T


,source,group,channel,whitening_proxy,resolution,target_dt_s,wdm_M,wdm_df_hz,first_positive_freq_hz,reaches_1e-5_hz,wdm_dt_tf_s,n_frequency_layers,n_signal_time_layers,signal_tf_pixels,t_whiten_signal_mean,t_whiten_signal_median,t_whiten_signal_max,t_whiten_offsource_mean,t_whiten_signal_over_offsource_mean
0,noisy,AET,A,T,dt2_M4096,2.0,4096,0.000061,0.000061,False,7692.307807,163,1,163,87679.824201,27580.552443,444882.858791,52.766928,1661.643519
1,noisy,AET,A,T,dt2_M2048,2.0,2048,0.000122,0.000122,False,4000.000060,81,2,162,52372.577237,71.343166,412707.480051,63.708762,822.062388
2,noisy,AET,A,T,dt1_M4096,1.0,4096,0.000122,0.000122,False,4000.000060,81,2,162,43859.394900,4463.199149,307136.940171,53.721607,816.420011
3,noisy,AET,E,T,dt2_M4096,2.0,4096,0.000061,0.000061,False,7692.307807,163,1,163,87252.417870,27240.907694,447151.443379,141.544266,616.432021
4,noisy,AET,A,T,dt1_M64,1.0,64,0.007812,0.007812,False,63.979528,1,84,84,20728.006558,642.511164,441145.374510,34.971777,592.706693
5,noisy,AET,A,T,dt1_M128,1.0,128,0.003906,0.003906,False,127.877240,2,42,84,27479.802152,652.142280,632879.553678,46.430268,591.851039
6,noisy,AET,A,T,dt1_M256,1.0,256,0.001953,0.001953,False,255.754480,5,21,105,30329.806077,795.067009,616957.004055,51.559414,588.249631
7,noisy,AET,A,T,dt2_M1024,2.0,1024,0.000244,0.000244,False,2040.816357,40,3,120,35352.004460,205.619084,384594.713176,62.915224,561.899050
8,noisy,AET,A,T,dt1_M2048,1.0,2048,0.000244,0.000244,False,2040.816357,40,3,120,32285.295975,490.080254,266444.806925,63.209074,510.769958
9,noisy,AET,A,T,dt1_M1024,1.0,1024,0.000488,0.000488,False,1020.408178,20,5,100,24988.271183,1259.140974,301384.659997,59.418015,420.550418



Combined best-pixel summary


,source,group,channel,best_pixel_power,best_pixel_log10_power,best_pixel_resolution,finite_bins
0,gw_only,links,31,9.940164e+293,293.997394,dt1_M4096,7594
1,gw_only,links,13,9.890289e+293,293.995209,dt1_M4096,7594
2,gw_only,links,12,9.312251e+293,293.969055,dt1_M4096,7594
3,gw_only,links,21,9.262942e+293,293.966749,dt1_M4096,7594
4,gw_only,links,32,5.900749e+293,293.770907,dt1_M4096,7594
5,gw_only,links,23,5.838519e+293,293.766303,dt1_M4096,7594
6,gw_only,AET,A,4.650737e+29,29.667522,dt2_M1024,7594
7,gw_only,XYZ,X,3.915861e+29,29.592827,dt2_M1024,7594
8,gw_only,AET,T,3.358893e+29,29.526196,dt2_M1024,7594
9,gw_only,XYZ,Z,2.460083e+29,29.390950,dt2_M1024,7594
